# 🤖 AI Agent Benchmark: Claude Code vs. Gemini CLI vs. Codex CLI

This notebook provides a rigorous framework to compare the performance of CLI-based AI agents. It sets up specific coding challenges, allows you to run your agent (interactively in the terminal), and then runs automated grading scripts to evaluate the results.

## 📊 The Metrics

For each task, we calculate a **Total Score (0-100)** based on:

1.  **Correctness (60%):** Do the unit tests pass? (`pytest`)
2.  **Code Quality (20%):** Is the code clean and idiomatic? (`pylint`)
3.  **Efficiency (20%):** Did the agent modify only what was necessary? (Line diff count)

## 🛠️ Setup

Run the cell below to install the necessary grading tools if they aren't already present.

In [1]:
!pip install pytest pylint pandas -q
import os
import subprocess
import pandas as pd
import shutil
import re

# Configuration
BENCHMARK_DIR = "tasks"
if not os.path.exists(BENCHMARK_DIR):
    os.makedirs(BENCHMARK_DIR)

print(f"📂 Benchmark directory set to: {os.path.abspath(BENCHMARK_DIR)}")

📂 Benchmark directory set to: /Users/plawanrath/Documents/GitHub-public/deep-learning-sandbox/benchmarks/tasks


## 🧠 Benchmark Engine

This class handles the setup, grading, and cleanup of tasks.

In [2]:
class BenchmarkEngine:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.results = []

    def create_task(self, task_name, files):
        """Creates a specific task folder with the provided files."""
        task_path = os.path.join(self.base_dir, task_name)
        if os.path.exists(task_path):
            shutil.rmtree(task_path)
        os.makedirs(task_path)
        
        print(f"🚀 Initializing Task: {task_name}")
        for filename, content in files.items():
            full_path = os.path.join(task_path, filename)
            with open(full_path, 'w') as f:
                f.write(content)
            print(f"  - Created {filename}")
        
        # Fixed: Correctly escaped newline for the print statement
        print(f"\n👉 ACTION: Open your terminal, cd to '{task_path}', and run your agent!")
        return task_path

    def revert_task(self, task_name, files):
        """Reverts a task to its original state by recreating all files."""
        print(f"🔄 Reverting task '{task_name}' to original state...")
        return self.create_task(task_name, files)

    def grade_task(self, task_name, agent_name):
        """Runs tests and linters to grade the agent's work."""
        task_path = os.path.join(self.base_dir, task_name)
        print(f"📝 Grading {agent_name} on {task_name}...")

        # 1. Run Tests (Pytest)
        try:
            test_result = subprocess.run(
                ['pytest', '.'],
                cwd=task_path,
                capture_output=True,
                text=True,
                timeout=10
            )
            tests_passed = test_result.returncode == 0
            test_output = test_result.stdout
        except Exception as e:
            tests_passed = False
            test_output = str(e)

        # 2. Run Linter (Pylint)
        try:
            # We look specifically for the implementation file (not the test file)
            py_files = [f for f in os.listdir(task_path) if f.endswith('.py') and not f.startswith('test_')]
            if py_files:
                lint_target = py_files[0]
                lint_result = subprocess.run(
                    ['pylint', lint_target, '--output-format=text'],
                    cwd=task_path,
                    capture_output=True,
                    text=True
                )
                # Extract score from "Your code has been rated at X.XX/10"
                score_match = re.search(r'rated at (-?[0-9.]+)/10', lint_result.stdout)
                lint_score = float(score_match.group(1)) if score_match else 0.0
            else:
                lint_score = 0.0
        except Exception:
            lint_score = 0.0

        # 3. Calculate Total
        test_score = 100 if tests_passed else 0
        final_score = (test_score * 0.6) + (max(0, lint_score) * 10 * 0.2) + 20 # Free 20 pts for efficiency (placeholder)

        result = {
            "Agent": agent_name,
            "Task": task_name,
            "Tests Passed": tests_passed,
            "Lint Score": round(lint_score, 2),
            "Total Score": round(final_score, 2)
        }
        self.results.append(result)
        return pd.DataFrame([result])

engine = BenchmarkEngine(BENCHMARK_DIR)

## 🧪 Task 1: The "Off-by-One" Bug (Debugging)

**Scenario:** A standard statistical utility library has a subtle bug in its median calculation logic and crashes on empty inputs.

**Agent Prompt:** *"Fix the bugs in `stats_util.py`. It should handle empty lists by returning None, and correctly calculate the median for both odd and even length lists."*

In [3]:
# FILE DEFINITIONS
task1_files = {
    "stats_util.py": '''
def calculate_median(numbers):
    """Calculates the median of a list of numbers."""
    numbers.sort()
    n = len(numbers)
    # Bug 1: No check for empty list (will index error)
    # Bug 2: Logic for even numbers is wrong, it just takes the lower middle
    return numbers[n // 2]
''',
    "test_stats.py": '''
import pytest
from stats_util import calculate_median

def test_median_odd():
    assert calculate_median([1, 3, 5]) == 3
    assert calculate_median([5, 1, 3]) == 3

def test_median_even():
    # For [1, 2, 3, 4], median is (2+3)/2 = 2.5
    assert calculate_median([1, 2, 3, 4]) == 2.5

def test_empty():
    assert calculate_median([]) is None
'''
}

# 1. SETUP
engine.create_task("task_01_debug", task1_files)

🚀 Initializing Task: task_01_debug
  - Created stats_util.py
  - Created test_stats.py

👉 ACTION: Open your terminal, cd to 'tasks/task_01_debug', and run your agent!


'tasks/task_01_debug'

### 🛑 STOP: Run **Gemini** now!
1. Open terminal: `cd tasks/task_01_debug`
2. Run **Gemini CLI** with prompt: *"Fix the bugs in stats_util.py. It should handle empty lists by returning None, and correctly calculate the median for both odd and even length lists."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [4]:
# 2. GRADE GEMINI
engine.grade_task("task_01_debug", agent_name="Gemini")

📝 Grading Gemini on task_01_debug...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Gemini,task_01_debug,True,6.25,92.5


### 🔄 REVERT: Restoring Clean State

Before running the second agent, we need to restore the task to its original state so both agents work with identical starting conditions.

In [6]:
# 3. REVERT
engine.revert_task("task_01_debug", task1_files)

🔄 Reverting task 'task_01_debug' to original state...
🚀 Initializing Task: task_01_debug
  - Created stats_util.py
  - Created test_stats.py

👉 ACTION: Open your terminal, cd to 'tasks/task_01_debug', and run your agent!


'tasks/task_01_debug'

### 🛑 STOP: Run **Claude Code** now!
1. Open terminal: `cd tasks/task_01_debug`
2. Run **Claude Code** with prompt: *"Fix the bugs in stats_util.py. It should handle empty lists by returning None, and correctly calculate the median for both odd and even length lists."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [7]:
# 4. GRADE CLAUDE
engine.grade_task("task_01_debug", agent_name="Claude")

📝 Grading Claude on task_01_debug...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Claude,task_01_debug,True,8.0,96.0


### 🔄 REVERT: Restoring Clean State

Before running the third agent, we need to restore the task to its original state so all agents work with identical starting conditions.

In [8]:
# 5. REVERT
engine.revert_task("task_01_debug", task1_files)

🔄 Reverting task 'task_01_debug' to original state...
🚀 Initializing Task: task_01_debug
  - Created stats_util.py
  - Created test_stats.py

👉 ACTION: Open your terminal, cd to 'tasks/task_01_debug', and run your agent!


'tasks/task_01_debug'

### 🛑 STOP: Run **Codex CLI** now!
1. Open terminal: `cd tasks/task_01_debug`
2. Run **Codex CLI** with prompt: *"Fix the bugs in stats_util.py. It should handle empty lists by returning None, and correctly calculate the median for both odd and even length lists."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [9]:
# 6. GRADE CODEX
engine.grade_task("task_01_debug", agent_name="Codex")

📝 Grading Codex on task_01_debug...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Codex,task_01_debug,True,8.89,97.78


## 🧪 Task 2: Feature Implementation (Data Parsing)

**Scenario:** We have a skeleton for a log parser, but the regex logic is missing.

**Agent Prompt:** *"Implement the `parse_log_line` function in `log_parser.py`. It needs to extract the Timestamp, Level (INFO/ERROR), and Message using regex. It should return a dictionary."*

In [13]:
task2_files = {
    "log_parser.py": '''
import re

def parse_log_line(line):
    """
    Parses a log line in the format: '[TIMESTAMP] LEVEL: Message'
    Example: '[2023-10-27 10:00:00] INFO: System started'
    
    Returns:
        dict: {'timestamp': str, 'level': str, 'message': str} or None if invalid
    """
    # TODO: Implement regex extraction
    return None
''',
    "test_parser.py": '''
from log_parser import parse_log_line

def test_valid_info():
    line = "[2023-10-27 10:00:00] INFO: System started"
    result = parse_log_line(line)
    assert result['timestamp'] == '2023-10-27 10:00:00'
    assert result['level'] == 'INFO'
    assert result['message'] == 'System started'

def test_valid_error():
    line = "[2023-11-01 09:15:22] ERROR: Connection failed"
    result = parse_log_line(line)
    assert result['level'] == 'ERROR'

def test_invalid_format():
    assert parse_log_line("Invalid log line") is None
'''
}

# 1. SETUP
engine.create_task("task_02_feature", task2_files)

🚀 Initializing Task: task_02_feature
  - Created log_parser.py
  - Created test_parser.py

👉 ACTION: Open your terminal, cd to 'tasks/task_02_feature', and run your agent!


'tasks/task_02_feature'

### 🛑 STOP: Run **Gemini** now!
1. Open terminal: `cd tasks/task_02_feature`
2. Run **Gemini CLI** with prompt: *"Implement the `parse_log_line` function in `log_parser.py`. It needs to extract the Timestamp, Level (INFO/ERROR), and Message using regex. It should return a dictionary."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [14]:
# 2. GRADE GEMINI
engine.grade_task("task_02_feature", agent_name="Gemini")

📝 Grading Gemini on task_02_feature...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Gemini,task_02_feature,True,8.57,97.14


### 🔄 REVERT: Restoring Clean State

Before running the second agent, we need to restore the task to its original state so both agents work with identical starting conditions.

In [15]:
# 3. REVERT
engine.revert_task("task_02_feature", task2_files)

🔄 Reverting task 'task_02_feature' to original state...
🚀 Initializing Task: task_02_feature
  - Created log_parser.py
  - Created test_parser.py

👉 ACTION: Open your terminal, cd to 'tasks/task_02_feature', and run your agent!


'tasks/task_02_feature'

### 🛑 STOP: Run **Claude Code** now!
1. Open terminal: `cd tasks/task_02_feature`
2. Run **Claude Code** with prompt: *"Implement the `parse_log_line` function in `log_parser.py`. It needs to extract the Timestamp, Level (INFO/ERROR), and Message using regex. It should return a dictionary."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [16]:
# 4. GRADE CLAUDE
engine.grade_task("task_02_feature", agent_name="Claude")

📝 Grading Claude on task_02_feature...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Claude,task_02_feature,True,8.57,97.14


### 🔄 REVERT: Restoring Clean State

Before running the third agent, we need to restore the task to its original state so all agents work with identical starting conditions.

In [17]:
# 5. REVERT
engine.revert_task("task_02_feature", task2_files)

🔄 Reverting task 'task_02_feature' to original state...
🚀 Initializing Task: task_02_feature
  - Created log_parser.py
  - Created test_parser.py

👉 ACTION: Open your terminal, cd to 'tasks/task_02_feature', and run your agent!


'tasks/task_02_feature'

### 🛑 STOP: Run **Codex CLI** now!
1. Open terminal: `cd tasks/task_02_feature`
2. Run **Codex CLI** with prompt: *"Implement the `parse_log_line` function in `log_parser.py`. It needs to extract the Timestamp, Level (INFO/ERROR), and Message using regex. It should return a dictionary."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [18]:
# 6. GRADE CODEX
engine.grade_task("task_02_feature", agent_name="Codex")

📝 Grading Codex on task_02_feature...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Codex,task_02_feature,True,8.57,97.14


## 🧪 Task 3: The Refactor (Code Quality)

**Scenario:** A piece of "spaghetti code" works but is unreadable. It uses single-letter variables and lacks type hints.

**Agent Prompt:** *"Refactor `inventory.py` to meet modern Python standards. Use meaningful variable names, add type hints, and add docstrings. Do NOT change the logic/functionality."*

In [19]:
task3_files = {
    "inventory.py": '''
def p(d, i, q):
    if i in d:
        d[i] += q
    else:
        d[i] = q
    return d

def c(d):
    t = 0
    for k in d:
        t += d[k]
    return t
''',
    "test_inventory.py": '''
import pytest
import inventory

def test_workflow():
    # Check if logic is preserved after refactoring
    # We assume the agent keeps the function names or we might need to update imports
    # Ideally, the agent should rename functions to 'process_inventory' etc.
    # For this test, we might need to inspect the module attributes if names change,
    # but let's assume for now strict API preservation OR intelligent alias.
    
    # Setup
    inv = {}
    
    # If functions were renamed, we try to find them, otherwise use originals
    add_func = getattr(inventory, 'add_item', None) or getattr(inventory, 'update_inventory', None) or inventory.p
    count_func = getattr(inventory, 'get_total', None) or getattr(inventory, 'count_items', None) or inventory.c

    add_func(inv, 'apple', 10)
    add_func(inv, 'banana', 5)
    add_func(inv, 'apple', 2)
    
    assert inv['apple'] == 12
    assert count_func(inv) == 17
'''
}

# 1. SETUP
engine.create_task("task_03_refactor", task3_files)

🚀 Initializing Task: task_03_refactor
  - Created inventory.py
  - Created test_inventory.py

👉 ACTION: Open your terminal, cd to 'tasks/task_03_refactor', and run your agent!


'tasks/task_03_refactor'

### 🛑 STOP: Run **Gemini** now!
1. Open terminal: `cd tasks/task_03_refactor`
2. Run **Gemini CLI** with prompt: *"Refactor `inventory.py` to meet modern Python standards. Use meaningful variable names, add type hints, and add docstrings. Do NOT change the logic/functionality."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [20]:
# 2. GRADE GEMINI
engine.grade_task("task_03_refactor", agent_name="Gemini")

📝 Grading Gemini on task_03_refactor...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Gemini,task_03_refactor,True,8.0,96.0


### 🔄 REVERT: Restoring Clean State

Before running the second agent, we need to restore the task to its original state so both agents work with identical starting conditions.

In [21]:
# 3. REVERT
engine.revert_task("task_03_refactor", task3_files)

🔄 Reverting task 'task_03_refactor' to original state...
🚀 Initializing Task: task_03_refactor
  - Created inventory.py
  - Created test_inventory.py

👉 ACTION: Open your terminal, cd to 'tasks/task_03_refactor', and run your agent!


'tasks/task_03_refactor'

### 🛑 STOP: Run **Claude Code** now!
1. Open terminal: `cd tasks/task_03_refactor`
2. Run **Claude Code** with prompt: *"Refactor `inventory.py` to meet modern Python standards. Use meaningful variable names, add type hints, and add docstrings. Do NOT change the logic/functionality."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [22]:
# 4. GRADE CLAUDE
engine.grade_task("task_03_refactor", agent_name="Claude")

📝 Grading Claude on task_03_refactor...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Claude,task_03_refactor,False,9.09,38.18


### 🔄 REVERT: Restoring Clean State

Before running the third agent, we need to restore the task to its original state so all agents work with identical starting conditions.

In [23]:
# 5. REVERT
engine.revert_task("task_03_refactor", task3_files)

🔄 Reverting task 'task_03_refactor' to original state...
🚀 Initializing Task: task_03_refactor
  - Created inventory.py
  - Created test_inventory.py

👉 ACTION: Open your terminal, cd to 'tasks/task_03_refactor', and run your agent!


'tasks/task_03_refactor'

### 🛑 STOP: Run **Codex CLI** now!
1. Open terminal: `cd tasks/task_03_refactor`
2. Run **Codex CLI** with prompt: *"Refactor `inventory.py` to meet modern Python standards. Use meaningful variable names, add type hints, and add docstrings. DO NOT change the logic/functionality."*
3. Wait for agent to finish.
4. Run the grade cell below.

In [24]:
# 6. GRADE CODEX
engine.grade_task("task_03_refactor", agent_name="Codex")

📝 Grading Codex on task_03_refactor...


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Codex,task_03_refactor,True,9.33,98.66


### Combine Results into a tabular view

In [25]:
df_results = pd.DataFrame(engine.results)
if not df_results.empty:
    print(df_results.groupby("Agent").mean(numeric_only=True))
    display(df_results)
else:
    print("No results yet!")

        Tests Passed  Lint Score  Total Score
Agent                                        
Claude      0.666667    8.553333    77.106667
Codex       1.000000    8.930000    97.860000
Gemini      1.000000    7.787500    95.575000


,Agent,Task,Tests Passed,Lint Score,Total Score
0,Gemini,task_01_debug,True,6.25,92.50
1,Claude,task_01_debug,True,8.00,96.00
2,Codex,task_01_debug,True,8.89,97.78
3,Gemini,task_02_feature,True,8.33,96.66
4,Gemini,task_02_feature,True,8.57,97.14
5,Claude,task_02_feature,True,8.57,97.14
6,Codex,task_02_feature,True,8.57,97.14
7,Gemini,task_03_refactor,True,8.00,96.00
8,Claude,task_03_refactor,False,9.09,38.18
9,Codex,task_03_refactor,True,9.33,98.66
